In [0]:
%run ./01_extracao_pedidos

In [0]:
import matplotlib.pyplot as plt

total = df_pedidos.count()
print(f"Linhas : {total}")
print(f"Colunas: {len(df_pedidos.columns)}")


def barras(pdf, x, y, titulo, rotulo_y, horizontal=False):
    """Grafico de barras a partir de um agregado pequeno ja trazido do Spark."""
    fig, ax = plt.subplots(figsize=(8, 3.5))
    if horizontal:
        ax.barh(pdf[x].astype(str), pdf[y], color="#2a6f97")
        ax.invert_yaxis()
        ax.set_xlabel(rotulo_y)
    else:
        ax.bar(pdf[x].astype(str), pdf[y], color="#2a6f97")
        ax.set_ylabel(rotulo_y)
        plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    ax.set_title(titulo, loc="left", fontsize=11)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()

In [0]:
por_lote = (
    df_pedidos.groupBy("dt_lote")
              .agg(F.count("*").alias("pedidos"),
                   F.round(F.sum("valor_total"), 2).alias("receita"))
              .orderBy("dt_lote")
)
display(por_lote)

pdf_lote = por_lote.toPandas()
pdf_lote["lote"] = pdf_lote["dt_lote"].dt.strftime("%d/%m %H:%M:%S")
barras(pdf_lote, "lote", "pedidos", "Pedidos por lote do gerador", "pedidos")

In [0]:
contagens = df_pedidos.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_pedidos.columns
]).first().asDict()

nulos = spark.createDataFrame(
    [(c, int(q), round(q / max(total, 1) * 100, 2)) for c, q in contagens.items()],
    ["coluna", "nulos", "pct_nulos"],
).orderBy(F.desc("nulos"))
display(nulos)
print(f"Colunas com ao menos um nulo: {sum(1 for v in contagens.values() if v)} de {len(contagens)}")

In [0]:
pk = CHAVE_PRIMARIA
nulos_pk   = df_pedidos.filter(F.col(pk).isNull()).count()
distintos  = df_pedidos.select(pk).distinct().count()
duplicados = total - distintos

print(f"=== Chave primaria: {pk} ===")
print(f"Registros  : {total}")
print(f"Nulos      : {nulos_pk}")
print(f"Distintos  : {distintos}")
print(f"Duplicados : {duplicados}")
print(f"Status     : {'SAUDAVEL' if nulos_pk == 0 and duplicados == 0 else 'ATENCAO'}")

if duplicados:
    display(df_pedidos.groupBy(pk).count().filter("count > 1").orderBy(F.desc("count")))

In [0]:
REGRAS = {
    "valor_total <= 0"                : F.col("valor_total") <= 0,
    "valor_frete > valor_total"       : F.col("valor_frete") > F.col("valor_total"),
    "status atualizado antes do pedido": F.col("dt_ultima_atualizacao_status") < F.col("dt_pedido"),
    "entrega prevista antes do pedido": F.col("dt_previsao_entrega") < F.col("dt_pedido"),
}

violacoes = df_pedidos.agg(*[
    F.sum(cond.cast("int")).alias(nome) for nome, cond in REGRAS.items()
]).first().asDict()

display(spark.createDataFrame(
    [(n, int(q), round(q / max(total, 1) * 100, 1)) for n, q in violacoes.items()],
    ["regra_violada", "pedidos", "pct"],
))

# Linhas que violam alguma regra, com a lista de regras que cada uma quebra.
viola_alguma = F.concat_ws(", ", *[F.when(c, F.lit(n)) for n, c in REGRAS.items()])
incoerentes = (
    df_pedidos.withColumn("regras_violadas", viola_alguma)
              .filter(F.col("regras_violadas") != "")
)
print(f"Pedidos com ao menos uma violacao: {incoerentes.count()} de {total}")
display(incoerentes.select(pk, "status_pedido", "valor_total", "valor_frete",
                           "dt_pedido", "dt_ultima_atualizacao_status", "regras_violadas"))

In [0]:
por_status = (
    df_pedidos.groupBy("status_pedido")
              .agg(F.count("*").alias("pedidos"),
                   F.round(F.avg("valor_total"), 2).alias("ticket_medio"))
              .withColumn("pct", F.round(F.col("pedidos") / total * 100, 1))
              .orderBy(F.desc("pedidos"))
)
display(por_status)
barras(por_status.toPandas(), "status_pedido", "pedidos", "Pedidos por status", "pedidos", horizontal=True)

In [0]:
receita_total = df_pedidos.agg(F.sum("valor_total")).first()[0]

por_pagamento = (
    df_pedidos.groupBy("metodo_pagamento")
              .agg(F.count("*").alias("pedidos"),
                   F.round(F.sum("valor_total"), 2).alias("receita"),
                   F.round(F.avg("valor_total"), 2).alias("ticket_medio"))
              .withColumn("pct_receita", F.round(F.col("receita") / F.lit(receita_total) * 100, 1))
              .orderBy(F.desc("receita"))
)
display(por_pagamento)
barras(por_pagamento.toPandas(), "metodo_pagamento", "receita", "Receita por meio de pagamento (R$)", "R$", horizontal=True)

In [0]:
estat = df_pedidos.agg(
    F.round(F.sum("valor_total"), 2).alias("receita"),
    F.round(F.avg("valor_total"), 2).alias("ticket_medio"),
    F.round(F.percentile_approx("valor_total", 0.5), 2).alias("mediana"),
    F.round(F.min("valor_total"), 2).alias("minimo"),
    F.round(F.max("valor_total"), 2).alias("maximo"),
    F.round(F.avg("valor_frete"), 2).alias("frete_medio"),
    F.round(F.percentile_approx(F.col("valor_frete") / F.col("valor_total"), 0.5) * 100, 1).alias("frete_pct_mediana"),
)
display(estat)

faixas = (
    df_pedidos.withColumn(
        "faixa",
        F.when(F.col("valor_total") <= 0, "0. <= 0")
         .when(F.col("valor_total") < 100, "1. até 100")
         .when(F.col("valor_total") < 250, "2. 100-250")
         .when(F.col("valor_total") < 500, "3. 250-500")
         .otherwise("4. 500+"),
    )
    .groupBy("faixa").agg(F.count("*").alias("pedidos"))
    .orderBy("faixa")
)
barras(faixas.toPandas(), "faixa", "pedidos", "Pedidos por faixa de valor (R$)", "pedidos")

In [0]:
prazo = df_pedidos.withColumn("prazo_dias", F.datediff("dt_previsao_entrega", "dt_pedido"))

display(prazo.agg(
    F.min("prazo_dias").alias("min_dias"),
    F.round(F.avg("prazo_dias"), 1).alias("media_dias"),
    F.percentile_approx("prazo_dias", 0.5).alias("mediana_dias"),
    F.max("prazo_dias").alias("max_dias"),
))

dist_prazo = prazo.groupBy("prazo_dias").agg(F.count("*").alias("pedidos")).orderBy("prazo_dias")
barras(dist_prazo.toPandas(), "prazo_dias", "pedidos", "Prazo de entrega prometido (dias)", "pedidos")

In [0]:
por_cliente = df_pedidos.groupBy("id_cliente").agg(F.count("*").alias("pedidos"))
clientes    = por_cliente.count()
recorrentes = por_cliente.filter("pedidos > 1").count()

print(f"Clientes distintos          : {clientes}")
print(f"Clientes com mais de 1 pedido: {recorrentes}")
print(f"Pedidos por cliente (media) : {total / max(clientes, 1):.2f}")